# 🌟 GoldyLoopAI — Eval Walkthrough

This notebook walks through the complete evaluation loop step-by-step.

**Steps:**
1. Load and validate the golden dataset
2. Run the RAG pipeline on every example
3. Score outputs with LLM-as-a-Judge
4. Analyze results by slice (difficulty, scenario)
5. Identify regressions and add new golden examples

> Make sure you have `OPENAI_API_KEY` set in your environment.

In [ ]:
import os, sys, json
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv('../.env')

print('✅ Environment loaded')

## Step 1: Load & Validate the Golden Dataset

In [ ]:
from golden_builder import validate_dataset

with open('../data/golden_dataset.json') as f:
    dataset = json.load(f)

report = validate_dataset(dataset)
print(f'Total examples  : {report["total_items"]}')
print(f'Valid           : {report["is_valid"]}')
print(f'Difficulty dist : {report["difficulty_distribution"]}')
print(f'Scenario tags   : {report["scenario_tags"]}')

df = pd.DataFrame([
    {'id': d['id'], 'difficulty': d['metadata']['difficulty'],
     'tag': d['metadata']['scenario_tag'], 'risk': d['metadata']['risk_level']}
    for d in dataset
])
df

## Step 2: Run the RAG Pipeline

In [ ]:
from app import run_on_golden_dataset

# Uses gpt-4o-mini by default (cheap and fast)
pipeline_outputs = run_on_golden_dataset('../data/golden_dataset.json', model='gpt-4o-mini')

print(f'\n✅ Generated {len(pipeline_outputs)} outputs')
print('\nSample output:')
print(f'Q: {pipeline_outputs[0]["input"]}')
print(f'A: {pipeline_outputs[0]["actual_output"]}')

## Step 3: Score with LLM-as-a-Judge

In [ ]:
from evaluator import run_evaluation, compute_summary

eval_results = run_evaluation(pipeline_outputs, judge_model='gpt-4o')
summary = compute_summary(eval_results)

print(f'\n📊 Summary')
print(f'Pass rate       : {summary["pass_rate"]}%')
print(f'Avg Correctness : {summary["avg_correctness"]}/5')
print(f'Avg Groundedness: {summary["avg_groundedness"]}/5')
print(f'Avg Completeness: {summary["avg_completeness"]}/5')
print(f'Avg Overall     : {summary["avg_overall"]}/5')

## Step 4: Slice Analysis

In [ ]:
# Build a DataFrame for slice analysis
valid = [r for r in eval_results if r.get('avg_score') is not None]
df_results = pd.DataFrame([{
    'id': r['id'],
    'difficulty': r['metadata']['difficulty'],
    'tag': r['metadata']['scenario_tag'],
    'risk': r['metadata']['risk_level'],
    'correctness': r['correctness'],
    'groundedness': r['groundedness'],
    'completeness': r['completeness'],
    'avg_score': r['avg_score'],
    'passed': r['passed'],
} for r in valid])

print('By Difficulty:')
print(df_results.groupby('difficulty')['avg_score'].agg(['mean','count']).round(2))
print('\nBy Scenario:')
print(df_results.groupby('tag')['avg_score'].agg(['mean','count']).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Difficulty breakdown
diff_scores = df_results.groupby('difficulty')['avg_score'].mean().reindex(['easy','medium','hard'])
diff_scores.plot(kind='bar', ax=axes[0], color=['#60a5fa','#f5c842','#f87171'], edgecolor='none', rot=0)
axes[0].set_title('Avg Score by Difficulty', fontweight='bold')
axes[0].set_ylim(0, 5)
axes[0].set_ylabel('Avg Score')

# Scenario breakdown
tag_scores = df_results.groupby('tag')['avg_score'].mean().sort_values()
tag_scores.plot(kind='barh', ax=axes[1], color='#f5c842', edgecolor='none')
axes[1].set_title('Avg Score by Scenario', fontweight='bold')
axes[1].set_xlim(0, 5)

plt.tight_layout()
plt.savefig('../data/slice_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## Step 5: Identify Failures & Close the Loop

In [ ]:
failures = df_results[~df_results['passed']]
print(f'❌ {len(failures)} failures detected:\n')
for _, row in failures.iterrows():
    result = next(r for r in valid if r['id'] == row['id'])
    print(f"ID: {row['id']} | difficulty: {row['difficulty']} | score: {row['avg_score']}")
    print(f"  Q: {result['input']}")
    print(f"  Reason: {result.get('reason','N/A')}")
    print()

### 🔁 The Loop

For each failure:
1. **Understand why it failed** — read the judge's `reason`
2. **Fix the prompt** in `app.py` or add guardrails
3. **Add it to the golden dataset** as a new example with stricter `expected_output`
4. **Re-run the pipeline** and verify the score improves

This is the **GoldyLoop** — every production failure becomes a stronger test case.